In [60]:
import numpy as np
import random
import math
import copy
import matplotlib.pyplot as plt
import pulp

def generer_matrices(t, difficulte=0.5, seed=1):
    """
    Génère deux matrices complètes :
    - MATRICES_P : distances positives symétriques
    - MATRICES_R : dangerosité (0 à 5), symétrique, influencée par 'difficulte'
    
    Paramètres :
    - t : taille de la matrice
    - difficulte : float entre 0 et 1 (0 = facile, 1 = très dangereux)
    - seed : pour reproductibilité
    
    Retour :
    - MATRICES_P (np.array)
    - MATRICES_R (np.array)
    """
    
    dist_min=10
    dist_max=80

    
    np.random.seed(seed)
    random.seed(seed)

    # -------------------------
    # MATRICE DES DISTANCES
    # -------------------------
    dist_matrix = np.random.randint(dist_min, dist_max + 1, size=(t, t))
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = (dist_matrix + dist_matrix.T) // 2  # symétrie

    # -------------------------
    # MATRICE DES COÛTS EN VIES
    # -------------------------
    MATRICES_R = np.zeros((t, t), dtype=int)

    for i in range(t):
        for j in range(i + 1, t):

            # probabilité d'avoir un coût élevé
            if random.random() < difficulte:
                # coût dangereux : 3 à 5
                cost = np.random.randint(3, 6)
            else:
                # coût faible : 0 à 2
                cost = np.random.randint(0, 3)

            MATRICES_R[i, j] = cost
            MATRICES_R[j, i] = cost

    return dist_matrix, MATRICES_R

P1 = generer_matrices(10)[0]
R1 = generer_matrices(10)[1]

P2 = generer_matrices(20)[0]
R2 = generer_matrices(20)[1]

P3 = generer_matrices(50)[0]
R3 = generer_matrices(50)[1]

MATRICES_P = [P1, P2, P3]
MATRICES_R = [R1, R2, R3]

# Affichage des matrices
for i in range(len(MATRICES_P)):
    print(f"Matrice P{i+1} (Distances) :\n{MATRICES_P[i]}\n")
    print(f"Matrice R{i+1} (Coûts en vies) :\n{MATRICES_R[i]}\n")


Matrice P1 (Distances) :
[[ 0 26 18 36 63 30 26 40 44 46]
 [26  0 47 39 37 28 52 76 26 18]
 [18 47  0 22 64 15 15 41 29 28]
 [36 39 22  0 41 68 50 11 62 32]
 [63 37 64 41  0 53 51 25 17 39]
 [30 28 15 68 53  0 41 64 51 20]
 [26 52 15 50 51 41  0 68 60 54]
 [40 76 41 11 25 64 68  0 49 43]
 [44 26 29 62 17 51 60 49  0 27]
 [46 18 28 32 39 20 54 43 27  0]]

Matrice R1 (Coûts en vies) :
[[0 3 0 2 5 5 3 1 2 5]
 [3 0 5 2 3 2 4 4 2 3]
 [0 5 0 2 0 5 4 1 1 5]
 [2 2 2 0 4 4 3 5 3 4]
 [5 3 0 4 0 3 4 4 3 4]
 [5 2 5 4 3 0 4 1 2 1]
 [3 4 4 3 4 4 0 3 0 2]
 [1 4 1 5 4 1 3 0 4 3]
 [2 2 1 3 3 2 0 4 0 0]
 [5 3 5 4 4 1 2 3 0 0]]

Matrice P2 (Distances) :
[[ 0 19 36 28 63 23 23 21 26 44 39 42 34 48 42 25 38 46 24 31]
 [19  0 53 38 48 37 17 54 30 50 54 26 46 75 43 53 46 32 50 43]
 [36 53  0 35 32 77 50 45 17 35 57 49 31 78 35 29 43 64 38 33]
 [28 38 35  0 35 55 62 66 51 40 50 65 33 16 27 47 50 39 35 37]
 [63 48 32 35  0 56 47 71 37 38 45 40 35 38 27 19 73 68 51 13]
 [23 37 77 55 56  0 45 35 50 33 45 41 49 2

# Métaheurestiques

In [40]:
def aco(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15, alpha=1, beta=2, rho=0.4, Q=50, num_ants=40, num_iterations=2000):
    """
    MATRICES_P : Matrice des distances (n x n)
    MATRICES_R : Matrice des coûts de vie (n x n)
    """
    n = MATRICES_P.shape[0]
    
    # Initialisation des phéromones et de l'heuristique
    pheromone = np.ones((n, n))
    # On évite la division par zéro
    heuristic = 1 / (MATRICES_P + 1e-10)
    np.fill_diagonal(heuristic, 0)

    # --- Fonctions internes ---
    def get_route_distance(route):
        return sum(MATRICES_P[route[i], route[i+1]] for i in range(len(route)-1))

    def evaluate_solution(routes, unvisited_left):
        distances = [get_route_distance(r) for r in routes]
        # Pénalité forte si des villes ne sont pas visitées
        if unvisited_left > 0:
            penalty = 10000 * unvisited_left
            return max(distances) + penalty, distances
        return max(distances), distances

    def choose_next(current, unvisited, life_left):
        # Filtrage par contrainte de vie
        feasible = [j for j in unvisited if MATRICES_R[current, j] <= life_left]
        if not feasible:
            return None

        weights = []
        total = 0
        for j in feasible:
            w = (pheromone[current, j] ** alpha) * (heuristic[current, j] ** beta)
            weights.append(w)
            total += w

        if total == 0:
            return random.choice(feasible)

        r = random.random() * total
        s = 0
        for j, w in zip(feasible, weights):
            s += w
            if s >= r:
                return j
        return feasible[-1]

    def build_solution():
        unvisited = set(range(1, n))
        routes = [[0] for _ in range(K)]
        life_left = [MAX_LIFE] * K
        vehicle = 0
        stuck_count = 0 

        while unvisited:
            current = routes[vehicle][-1]
            nxt = choose_next(current, list(unvisited), life_left[vehicle])

            if nxt is None:
                if routes[vehicle][-1] != 0:
                    routes[vehicle].append(0)
                
                vehicle = (vehicle + 1) % K
                stuck_count += 1
                if stuck_count >= K:
                    break
                continue

            stuck_count = 0
            routes[vehicle].append(nxt)
            life_left[vehicle] -= MATRICES_R[current, nxt]
            unvisited.remove(nxt)
            vehicle = (vehicle + 1) % K

        for v in range(K):
            if routes[v][-1] != 0:
                routes[v].append(0)

        return routes, len(unvisited)

    # --- Boucle ACO ---
    best_solution = None
    best_cost = float("inf")
    best_distances = []
    best_unvisited = n

    for it in range(num_iterations):
        solutions = []
        for _ in range(num_ants):
            routes, unvisited_left = build_solution()
            cost, distances = evaluate_solution(routes, unvisited_left)
            solutions.append((routes, cost, distances, unvisited_left))

            if cost < best_cost:
                best_cost = cost
                best_solution = routes
                best_distances = distances
                best_unvisited = unvisited_left

        # Évaporation
        pheromone *= (1 - rho)

        # Mise à jour des phéromones (Top 3 solutions valides)
        solutions.sort(key=lambda x: x[1])
        for routes, cost, distances, unvisited_left in solutions[:3]:
            if unvisited_left > 0:
                continue
            deposit = Q / (cost + 1e-10)
            for route in routes:
                for i in range(len(route)-1):
                    a, b = route[i], route[i+1]
                    pheromone[a, b] += deposit
                    pheromone[b, a] += deposit

    # --- Préparation du retour ---
    is_feasible = (best_unvisited == 0)
    
    # Transformation des routes en dictionnaire "1", "2", "3"...
    cycles_vaisseaux = {}
    if best_solution:
        for idx, r in enumerate(best_solution):
            cycles_vaisseaux[str(idx + 1)] = r
    else:
        # Cas où aucune solution n'a été construite du tout
        cycles_vaisseaux = {str(i+1): [0, 0] for i in range(K)}

    max_dist = max(best_distances) if best_distances else 0.0

    return is_feasible, cycles_vaisseaux, max_dist

In [62]:
def vns_search(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15, MAX_ITER=2000, MAX_VOISIN=3):
    """
    Exécute la recherche de voisinage variable (VNS).
    
    Retourne:
    - is_feasible (bool)
    - cycles_vaisseaux (dict)
    - max_distance (float)
    """
    
    # ============================================================
    # ---------------------- OUTILS INTERNES ---------------------
    # ============================================================
    
    def distance_tournee(route, matrice_p):
        return sum(matrice_p[route[i]][route[i+1]] for i in range(len(route)-1))

    def evaluer(routes, matrice_p, matrice_r):
        """Retourne la distance max (Min-Max)."""
        return max(distance_tournee(r, matrice_p) for r in routes)

    def route_lives(route, matrice_r):
        return sum(matrice_r[route[i]][route[i+1]] for i in range(len(route)-1))

    def est_valide(routes, matrice_p, matrice_r):
        """Vérifie les contraintes de vie et la visite de tous les nœuds."""
        N = len(matrice_p)
        visited = []
        for r in routes:
            if route_lives(r, matrice_r) > MAX_LIFE:
                return False
            visited.extend(r[1:-1])
        return sorted(visited) == list(range(1, N))

    def solution_initiale(planetes, matrice_p, matrice_r):
        """Répartition simple pour démarrer."""
        routes = [[0] for _ in range(K)]
        for i, p in enumerate(planetes):
            routes[i % K].append(p)
        for r in routes:
            r.append(0)
        return routes

    # ============================================================
    # ----------------------- VOISINAGES -------------------------
    # ============================================================

    def voisin_2opt(solution: list[list], matrice_p) -> list[list]:
        """
        2opt : choisit le vaisseau avec la tournée la plus longue et inverse un segment aléatoire de sa tournée.
        
        Avant :
            Tournée de départ (Vaisseau 1 le plus long) : [0, 1, 2, 3, 0]
            On tire i=1 et j=3 → segment [1, 2, 3]
            On inverse ce segment → [3, 2, 1]
        Après :
            Tournée modifiée : [0, 3, 2, 1, 0]
            2 arêtes supprimés : (0 → 1) et (2 → 3)
            2 arêtes ajoutés : (0 → 3) et (1 → 0)
        """
        sol = copy.deepcopy(solution)                                           # Deep copy pour éviter de modifier la solution originale
        k_max = max(range(K), key=lambda k: distance_tournee(sol[k], matrice_p))           # Trouve le vaisseau avec la tournée la plus longue
        t = sol[k_max]                                                          # Récupère la tournée du vaisseau le plus long
        if len(t) <= 3:                                                         # Si la tournée a moins de 2 planètes (hors dépôts), on ne peut pas faire de 2-opt
            return sol
        
        i, j = sorted(random.sample(range(1, len(t)-1), 2))                     # Choisit deux indices aléatoires i < j dans la tournée (on évite les dépôts)
        t[i:j+1] = t[i:j+1][::-1]                                               # Inverse le segment entre i et j
        return sol                                                              # Retourne la nouvelle solution voisine

    def voisin_relocate(solution: list[list], matrice_p) -> list[list]:
        """Relocate : prend une planète du vaisseau le plus chargé et la déplace vers un autre vaisseau choisi aléatoirement, à une position aléatoire.

        Avant :
            Vaisseau 1 (le plus long) : [1, 2, 3, 4, 5]
            Vaisseau 2                : [6, 7]
            Vaisseau 3                : [8, 9]

            On tire idx=2 → planète 3
            On la retire du vaisseau 1 → [1, 2, 4, 5]
            On choisit vaisseau 2 comme destination, pos=1

        Après :
            Vaisseau 1 : [1, 2, 4, 5]
            Vaisseau 2 : [6, 3, 7]      ← planète 3 insérée en pos 1
            Vaisseau 3 : [8, 9]
        """
        sol = copy.deepcopy(solution)
        k_max = max(range(K), key=lambda k: distance_tournee(sol[k], matrice_p))
        if len(sol[k_max]) <= 2:                                                # Si la tournée du vaisseau le plus long est vide, on ne peut pas relocaliser
            return sol
        
        idx = random.randrange(1, len(sol[k_max])-1)                            # Choisit une planète aléatoire dans la tournée la plus longue (on évite le dépôt)
        planete = sol[k_max].pop(idx)                                           # Retire la planète de la tournée du vaisseau le plus long

        autres = [k for k in range(K) if k != k_max]                            # Liste des autres vaisseaux, sans le vaisseau le plus long
        k_dest = random.choice(autres)                                          # Choisit un vaisseau de destination aléatoire parmi les autres
        pos = random.randint(1, len(sol[k_dest])-1)                             # Choisit une position aléatoire dans la tournée du vaisseau de destination
        sol[k_dest].insert(pos, planete)
        return sol

    def voisin_swap_inter(solution: list[list]) -> list[list]:
        """Swap inter-tournées : échange une planète entre deux vaisseaux différents.
        
        Avant :
            Vaisseau 1 : [1, 2, 3, 4]
            Vaisseau 2 : [5, 6, 7]

        On tire i1=1 (planète 2) et i2=2 (planète 7)
            → on les échange

        Après :
            Vaisseau 1 : [1, 7, 3, 4]
            Vaisseau 2 : [5, 6, 2]
        """
        sol = copy.deepcopy(solution)

        k1, k2 = random.sample(range(K), 2)                                     # On tire 2 vaisseaux différents au hasard (≠ relocate qui cible toujours le plus long)
        if len(sol[k1]) <= 2 or len(sol[k2]) <= 2:                             # Les deux tournées doivent avoir au moins 1 planète
            return sol
        
        i1 = random.randrange(1, len(sol[k1])-1)                                # On tire un indice aléatoire dans chaque tournée
        i2 = random.randrange(1, len(sol[k2])-1)

        sol[k1][i1], sol[k2][i2] = sol[k2][i2], sol[k1][i1]                     # On échange les planètes
        return sol

    VOISINAGES = [voisin_2opt, voisin_relocate, voisin_swap_inter]

    # ============================================================
    # --------------------------- VNS ----------------------------
    # ============================================================

    # Phase d'initialisation, si la solution initiale viole une contrainte de risque, on retente
    planetes = list(range(1, len(MATRICES_P)))
    sol_courante = solution_initiale(planetes, MATRICES_P, MATRICES_R)                     # Solution courante (initiale)
    tentatives = 0
    while not est_valide(sol_courante, MATRICES_P, MATRICES_R) and tentatives < 50:
        sol_courante = solution_initiale(planetes, MATRICES_P, MATRICES_R)
        tentatives += 1

    meilleure_sol = copy.deepcopy(sol_courante)                     # Meilleure solution globale
    meilleure_val = evaluer(sol_courante, MATRICES_P, MATRICES_R)                          # Valeur de la meilleure solution (distance max)
    historique = [meilleure_val]                                   # Historique de l'évolution de la meilleure valeur (pour analyse) 

    # Phase 2 — boucle principale (2000 itérations max)          

    for iteration in range(MAX_ITER):
        k_voisin = 0                                                # On repart toujours du voisinage 0 (2-opt)
        """
        À chaque itération on a une boucle interne qui parcourt les voisinages dans l'ordre :
            k_voisin=0 → 2-opt
            k_voisin=1 → relocate
            k_voisin=2 → swap inter
        """

        # Phase 3 — perturbation + recherche locale

        while k_voisin < MAX_VOISIN:                                # Tant que tous les voisinages n'ont pas été explorés

            if k_voisin == 2:  # swap n'a pas besoin de matrice_p
                voisine = VOISINAGES[k_voisin](sol_courante)
            else:
                voisine = VOISINAGES[k_voisin](sol_courante, MATRICES_P)

            for _ in range(10):                                     # Recherche locale légère : on tente 10 améliorations 2-opt sur ce voisin
                candidat = voisin_2opt(voisine, MATRICES_P)
                if evaluer(candidat, MATRICES_P, MATRICES_R) < evaluer(voisine, MATRICES_P, MATRICES_R):
                    voisine = candidat                              # On garde si c'est mieux (10 est un paramètre à ajuster)

            # Phase 4 - Décision

            val_voisine = evaluer(voisine, MATRICES_P, MATRICES_R)

            if val_voisine < evaluer(sol_courante, MATRICES_P, MATRICES_R):
                sol_courante = voisine                              # On accepte la solution voisine comme nouvelle solution courante
                k_voisin = 0                                        # On repart du voisinage 0 => on exploite à fond les améliorations 2-opt avant de passer aux autres.
                if val_voisine < meilleure_val:
                    meilleure_sol = copy.deepcopy(voisine)          # Nouvelle meilleure solution globale
                    meilleure_val = val_voisine
            else:
                k_voisin += 1                                       # Pas d'amélioration => on passe au voisinage suivant

        historique.append(meilleure_val)

    # Formatage de sortie standardisé
    is_feasible = est_valide(meilleure_sol, MATRICES_P, MATRICES_R)
    cycles_vaisseaux = {str(i + 1): route for i, route in enumerate(meilleure_sol)}

    return is_feasible, cycles_vaisseaux, meilleure_val

In [63]:
def genetic_algorithm(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15,
                      pop_size=150, generations=400, p_mutate=0.2):

    n = MATRICES_P.shape[0]

    # Gestion du MAX_LIFE
    if isinstance(MAX_LIFE, (int, float)):
        R_max = [MAX_LIFE] * K
    else:
        R_max = MAX_LIFE

    # ---------------------------------------------------------
    # 1) Représentation : permutation unique des nœuds 1..n-1
    # ---------------------------------------------------------
    def create_individual():
        perm = list(range(1, n))
        random.shuffle(perm)
        return perm

    # ---------------------------------------------------------
    # 2) Conversion permutation -> K routes équilibrées
    # ---------------------------------------------------------
    def perm_to_routes(perm):
        routes = [[] for _ in range(K)]
        for i, node in enumerate(perm):
            routes[i % K].append(node)
        return [[0] + r + [0] for r in routes]

    # ---------------------------------------------------------
    # 3) Distance et risque d'une route
    # ---------------------------------------------------------
    def route_distance(route):
        return sum(MATRICES_P[route[i], route[i+1]] for i in range(len(route)-1))

    def route_risk(route):
        return sum(MATRICES_R[route[i], route[i+1]] for i in range(len(route)-1))

    # ---------------------------------------------------------
    # 4) Fitness = max distance + pénalité risque
    # ---------------------------------------------------------
    def fitness(perm):
        routes = perm_to_routes(perm)
        distances = [route_distance(r) for r in routes]
        risks = [route_risk(r) for r in routes]

        penalty = 0
        for k in range(K):
            if risks[k] > R_max[k]:
                penalty += 5000 * (risks[k] - R_max[k] + 1)

        return max(distances) + penalty

    # ---------------------------------------------------------
    # 5) Sélection par tournoi
    # ---------------------------------------------------------
    def selection(pop):
        i, j = random.sample(range(len(pop)), 2)
        return pop[i] if fitness(pop[i]) < fitness(pop[j]) else pop[j]

    # ---------------------------------------------------------
    # 6) Crossover OX (Order Crossover)
    # ---------------------------------------------------------
    def crossover(p1, p2):
        size = len(p1)
        a, b = sorted(random.sample(range(size), 2))

        child = [None] * size
        child[a:b] = p1[a:b]

        fill = [x for x in p2 if x not in child]
        idx = 0
        for i in range(size):
            if child[i] is None:
                child[i] = fill[idx]
                idx += 1

        return child

    # ---------------------------------------------------------
    # 7) Mutation : swap ou inversion
    # ---------------------------------------------------------
    def mutate(perm):
        if random.random() < p_mutate:
            perm = perm[:]
            if random.random() < 0.5:
                # swap
                i, j = random.sample(range(len(perm)), 2)
                perm[i], perm[j] = perm[j], perm[i]
            else:
                # inversion
                i, j = sorted(random.sample(range(len(perm)), 2))
                perm[i:j] = reversed(perm[i:j])
        return perm

    # ---------------------------------------------------------
    # 8) Boucle principale avec élitisme
    # ---------------------------------------------------------
    population = [create_individual() for _ in range(pop_size)]
    best = min(population, key=fitness)

    for _ in range(generations):
        new_pop = [best]  # élitisme

        while len(new_pop) < pop_size:
            p1 = selection(population)
            p2 = selection(population)
            child = crossover(p1, p2)
            child = mutate(child)
            new_pop.append(child)

        population = new_pop
        current_best = min(population, key=fitness)
        if fitness(current_best) < fitness(best):
            best = current_best

    # ---------------------------------------------------------
    # 9) Résultat final
    # ---------------------------------------------------------
    routes = perm_to_routes(best)
    risks = [route_risk(r) for r in routes]
    feasible = all(risks[k] <= R_max[k] for k in range(K))
    max_dist = max(route_distance(r) for r in routes)

    cycles = {str(i+1): r for i, r in enumerate(routes)}

    return feasible, cycles, max_dist

In [64]:
def tabu_search(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15, iterations=300, tenure=7):
    """
    Exécute une recherche tabou pour le problème de tournées de véhicules.
    
    Retourne:
    - is_feasible (bool)
    - cycles_vaisseaux (dict)
    - max_distance (float)
    """
    N = len(MATRICES_P)
    START = 0

    # --- Outils internes ---
    def get_route_distance(route):
        return sum(MATRICES_P[route[i]][route[i + 1]] for i in range(len(route) - 1))

    def get_route_lives(route):
        return sum(MATRICES_R[route[i]][route[i + 1]] for i in range(len(route) - 1))

    def check_feasibility(routes):
        visited = []
        for r in routes:
            if get_route_lives(r) > MAX_LIFE:
                return False
            visited.extend(r[1:-1])
        return sorted(visited) == list(range(1, N))

    def evaluate(routes, penalty=1000):
        distances = [get_route_distance(r) for r in routes]
        max_dist = max(distances)
        violation = 0
        for r in routes:
            lives = get_route_lives(r)
            if lives > MAX_LIFE:
                violation += (lives - MAX_LIFE)
        return max_dist + penalty * violation

    def get_neighbors(routes):
        neigh = []
        for a in range(len(routes)):
            for b in range(len(routes)):
                if a == b: continue
                ra, rb = routes[a], routes[b]
                for i in range(1, len(ra) - 1):
                    node = ra[i]
                    new_ra = ra[:i] + ra[i+1:]
                    for j in range(1, len(rb)):
                        new_rb = rb[:j] + [node] + rb[j:]
                        new_sol = [list(r) for r in routes] # Copie rapide
                        new_sol[a] = new_ra
                        new_sol[b] = new_rb
                        neigh.append((new_sol, (node, a, b)))
        return neigh

    # --- Initialisation ---
    customers = list(range(1, N))
    current_sol = [[START] for _ in range(K)]
    for i, c in enumerate(customers):
        current_sol[i % K].append(c)
    for i in range(K):
        current_sol[i].append(START)

    best_sol = deepcopy(current_sol)
    best_score = evaluate(best_sol)
    tabu_list = {} # move -> iteration d'expiration

    # --- Boucle principale ---
    for it in range(iterations):
        cand_best, cand_move, cand_score = None, None, math.inf

        for sol, move in get_neighbors(current_sol):
            score = evaluate(sol)
            
            # Critère d'aspiration : on ignore le tabou si on bat le record absolu
            is_tabu = move in tabu_list and tabu_list[move] > it
            if is_tabu and score >= best_score:
                continue

            if score < cand_score:
                cand_score, cand_best, cand_move = score, sol, move

        if cand_best is None:
            break

        current_sol = cand_best
        tabu_list[cand_move] = it + tenure

        # Mise à jour du meilleur score global si la solution est réalisable
        if cand_score < best_score and check_feasibility(current_sol):
            best_sol = deepcopy(current_sol)
            best_score = cand_score

    # --- Formatage de la sortie ---
    is_feasible = check_feasibility(best_sol)
    cycles_vaisseaux = {str(i + 1): route for i, route in enumerate(best_sol)}
    
    # On recalcule la distance max réelle sans pénalité
    max_distance = max(get_route_distance(r) for r in best_sol)

    return is_feasible, cycles_vaisseaux, max_distance

# MILP

In [65]:
def solve_milp_exact(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15):
    """
    Version corrigée du solveur exact MILP.
    """
    N = len(MATRICES_P)
    START = 0
    clients = [i for i in range(N) if i != START]
    
    model = pulp.LpProblem("MinMax_kTSP_Exact", pulp.LpMinimize)

    # --- Variables ---
    # On crée toutes les combinaisons i, j pour x
    x = pulp.LpVariable.dicts("x", (range(K), range(N), range(N)), cat="Binary")
    y = pulp.LpVariable.dicts("y", (range(K), range(N)), cat="Binary")
    u = pulp.LpVariable.dicts("u", (range(K), clients), lowBound=1, upBound=len(clients))
    z = pulp.LpVariable("z", lowBound=0)

    model += z

    # --- Contraintes ---
    for i in clients:
        model += pulp.lpSum(y[v][i] for v in range(K)) == 1

    for v in range(K):
        for i in clients:
            model += pulp.lpSum(x[v][j][i] for j in range(N) if j != i) == y[v][i]
            model += pulp.lpSum(x[v][i][j] for j in range(N) if j != i) == y[v][i]

    for v in range(K):
        # Départ et retour obligatoires si le véhicule est utilisé
        model += pulp.lpSum(x[v][START][j] for j in clients) <= 1
        model += pulp.lpSum(x[v][j][START] for j in clients) == pulp.lpSum(x[v][START][j] for j in clients)

    for v in range(K):
        model += pulp.lpSum(MATRICES_P[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= z

    for v in range(K):
        model += pulp.lpSum(MATRICES_R[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= MAX_LIFE

    # MTZ Anti-sous-tours
    M = len(clients)
    for v in range(K):
        for i in clients:
            for j in clients:
                if i != j:
                    model += u[v][i] - u[v][j] + M * x[v][i][j] <= M - 1

    # Résolution
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    
    is_feasible = pulp.LpStatus[model.status] == 'Optimal'
    max_distance = pulp.value(z) if is_feasible and pulp.value(z) is not None else 0.0
    
    # --- Reconstruction des cycles (CORRIGÉE) ---
    cycles_vaisseaux = {}
    if is_feasible:
        for v in range(K):
            route = [START]
            curr = START
            # On limite la boucle pour éviter les tours infinis au cas où
            for _ in range(N):
                next_node = None
                for j in range(N):
                    if curr != j:
                        # pulp.value peut renvoyer None si la variable n'est pas utilisée
                        val = pulp.value(x[v][curr][j])
                        if val is not None and val > 0.5:
                            next_node = j
                            break
                
                if next_node is None or next_node == START:
                    break
                
                route.append(next_node)
                curr = next_node
            
            route.append(START)
            cycles_vaisseaux[str(v + 1)] = route
    else:
        cycles_vaisseaux = {str(v + 1): [0, 0] for v in range(K)}

    return is_feasible, cycles_vaisseaux, max_distance

In [66]:
def solve_strong_lp(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15):
    """
    Borne inférieure via relaxation linéaire.
    """
    N = len(MATRICES_P)
    model = pulp.LpProblem("Strong_LP", pulp.LpMinimize)

    # Variables continues entre 0 et 1 (Relaxation)
    x = pulp.LpVariable.dicts("x", (range(K), range(N), range(N)), 0, 1, cat="Continuous")
    y = pulp.LpVariable.dicts("y", (range(K), range(N)), 0, 1, cat="Continuous")
    z = pulp.LpVariable("z", lowBound=0)

    model += z

    # Chaque client visité au total une fois (somme des fractions = 1)
    for i in range(1, N):
        model += pulp.lpSum(y[v][i] for v in range(K)) == 1

    # Flux
    for v in range(K):
        for i in range(1, N):
            model += pulp.lpSum(x[v][i][j] for j in range(N) if j != i) == y[v][i]
            model += pulp.lpSum(x[v][j][i] for j in range(N) if j != i) == y[v][i]

    # Dépôt
    for v in range(K):
        model += pulp.lpSum(x[v][0][j] for j in range(1, N)) <= 1
        model += pulp.lpSum(x[v][j][0] for j in range(1, N)) <= 1

    # Distance max
    for v in range(K):
        model += pulp.lpSum(MATRICES_P[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= z

    # Vies
    for v in range(K):
        model += pulp.lpSum(MATRICES_R[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= MAX_LIFE

    model.solve(pulp.PULP_CBC_CMD(msg=False))

    is_feasible = pulp.LpStatus[model.status] == 'Optimal'
    
    # Pas de cycles réels en LP relaxation car les routes sont fractionnaires
    cycles_vaisseaux = {str(v + 1): [] for v in range(K)}
    max_dist = pulp.value(model.objective) if is_feasible else 0.0

    return is_feasible, cycles_vaisseaux, max_dist

# AFFICHAGE

In [68]:
NB_VEHICULE=3
MAX_LIFE=15


print("faisabilite, Cycles, Distance max")
print(aco(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, alpha=1, beta=2, rho=0.4, Q=50, num_ants=40, num_iterations=200))
print(vns_search(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, MAX_ITER=2000, MAX_VOISIN=3))
print(genetic_algorithm(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, pop_size=50, generations=50, p_mutate=0.2))
print(tabu_search(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, iterations=300, tenure=7))

#print(solve_milp_exact(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE)) #Attention il prend du temps lui !
print(solve_strong_lp(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE))


faisabilite, Cycles, Distance max
(True, {'1': [0, 3, 9, 5, 0], '2': [0, 1, 4, 8, 0], '3': [0, 7, 2, 6, 0]}, 124)
(True, {'1': [0, 8, 4, 0], '2': [0, 5, 9, 1, 0], '3': [0, 6, 2, 3, 7, 0]}, 124)
(True, {'1': [0, 8, 4, 7, 0], '2': [0, 6, 5, 2, 0], '3': [0, 3, 9, 1, 0]}, 126)
(True, {'1': [0, 1, 4, 8, 0], '2': [0, 5, 9, 0], '3': [0, 7, 3, 2, 6, 0]}, 124)
(True, {'1': [], '2': [], '3': []}, 50.666667)
